# Pilot · build the 500-answer bank (250 math + 250 code)

Runtime: **A100**. One fresh runtime. Needs two Colab secrets (key icon on the left):
- `HF_TOKEN`, with access to Llama and Gemma (their tokenizers are used for the length filter);
- `OPENROUTER_API_KEY`, for GPT-4.1-mini. The script stops at a **$5** total and retries 429s with backoff.

Steps: generate with Phi-4-mini, Granite-3.3-8B, OLMo-2-7B (vLLM) and GPT-4.1-mini (OpenRouter) → label math (boxed answer + Math-Verify) and code (EvalPlus base+plus tests, run twice) → matched pairs per (problem, generator) → drop answers too long for any verifier → draw 125 pairs per domain under the PLAN.md §2 caps. About an hour.

In [1]:
BRANCH = "feature/pilot-bank"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q -b {BRANCH} https://github.com/Manas-33/CSCI-544-Course-Project.git /content/project
%cd /content/project
!git log --oneline -1
!pip install -q -r bank/requirements.txt evalplus pyarrow
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))
import os
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
/content/project
39aa05e (HEAD -> feature/pilot-bank, origin/feature/pilot-bank) Add the pilot analysis: gates, AUROC intervals, pairs at matched TAR, figures
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is inc

## 1. Open generators (vLLM): math then code for each
Each call is its own process, so GPU memory is freed between models.

In [2]:
for d in ("math", "code"):
    !python -m bank.generate --questions bank/pilot/questions_{d}.jsonl --out bank/pilot/gen/phi4mini_{d}/generations.jsonl --model microsoft/Phi-4-mini-instruct --tag phi4mini
    !python -m bank.generate --questions bank/pilot/questions_{d}.jsonl --out bank/pilot/gen/granite33_{d}/generations.jsonl --model ibm-granite/granite-3.3-8b-instruct --tag granite33
    !python -m bank.generate --questions bank/pilot/questions_{d}.jsonl --out bank/pilot/gen/olmo2_{d}/generations.jsonl --model allenai/OLMo-2-1124-7B-Instruct --tag olmo2

Using dtype bfloat16, microsoft/Phi-4-mini-instruct at revision cfbefacb99257ffa30c83adab238a50856ac3083, math settings
(MainProcess pid=2753) INFO 10-27 07:22:27 [api_utils.py:286] non-default args: {'dtype': 'bfloat16', 'seed': 544, 'max_model_len': 4096, 'disable_log_stats': True, 'revision': 'cfbefacb99257ffa30c83adab238a50856ac3083', 'model': 'microsoft/Phi-4-mini-instruct'}
config.json: 100% 2.50k/2.50k [00:00<00:00, 1.75MB/s]
[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
(MainProcess pid=2753) INFO 10-27 07:22:43 [model.py:733] Resolved architecture: Phi3ForCausalLM
(MainProcess pid=2753) INFO 10-27 07:22:43 [model.py:2152] Using max model l

## 2. GPT-4.1-mini through OpenRouter
Resumable: if it stops on rate limits, rerun the cell and it continues where it left off. The spend so far is in `bank/pilot/api_spend.json`.

In [3]:
for d in ("math", "code"):
    !python -m bank.generate_api --questions bank/pilot/questions_{d}.jsonl --out bank/pilot/gen/gpt41mini_{d}/generations.jsonl
!cat bank/pilot/api_spend.json | head -5

0 answers already present, 1200 to request; spent so far $0.000 of $5.00
100/1200 {'ok': 100, 'failed': 0, 'over_budget': 0} spent $0.037
200/1200 {'ok': 200, 'failed': 0, 'over_budget': 0} spent $0.074
300/1200 {'ok': 300, 'failed': 0, 'over_budget': 0} spent $0.111
400/1200 {'ok': 400, 'failed': 0, 'over_budget': 0} spent $0.149
500/1200 {'ok': 500, 'failed': 0, 'over_budget': 0} spent $0.189
600/1200 {'ok': 600, 'failed': 0, 'over_budget': 0} spent $0.222
700/1200 {'ok': 700, 'failed': 0, 'over_budget': 0} spent $0.314
800/1200 {'ok': 800, 'failed': 0, 'over_budget': 0} spent $0.406
900/1200 {'ok': 900, 'failed': 0, 'over_budget': 0} spent $0.504
1000/1200 {'ok': 1000, 'failed': 0, 'over_budget': 0} spent $0.601
1100/1200 {'ok': 1100, 'failed': 0, 'over_budget': 0} spent $0.685
1200/1200 {'ok': 1200, 'failed': 0, 'over_budget': 0} spent $0.774
{'ok': 1200, 'failed': 0, 'over_budget': 0}; 1200 answers in bank/pilot/gen/gpt41mini_math/generations.jsonl; total spend $0.774
0 answers al

## 3. Merge each domain's answers

In [4]:
import json, glob, os
for d in ("math", "code"):
    rows = []
    for path in sorted(glob.glob(f"bank/pilot/gen/*_{d}/generations.jsonl")):
        rows += [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
        print(path, sum(1 for _ in open(path, encoding="utf-8")))
    os.makedirs(f"bank/pilot/{d}", exist_ok=True)
    with open(f"bank/pilot/{d}/generations.jsonl", "w", encoding="utf-8", newline="\n") as f:
        for r in sorted(rows, key=lambda r: r["candidate_id"]):
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(d, len(rows), "answers")

bank/pilot/gen/gpt41mini_math/generations.jsonl 1200
bank/pilot/gen/granite33_math/generations.jsonl 1200
bank/pilot/gen/olmo2_math/generations.jsonl 1200
bank/pilot/gen/phi4mini_math/generations.jsonl 1200
math 4800 answers
bank/pilot/gen/gpt41mini_code/generations.jsonl 1200
bank/pilot/gen/granite33_code/generations.jsonl 1200
bank/pilot/gen/olmo2_code/generations.jsonl 1200
bank/pilot/gen/phi4mini_code/generations.jsonl 1200
code 4800 answers


## 4. Labels
Math: the boxed answer and Math-Verify must agree. Code: EvalPlus's base and plus tests, each program twice (about 20 minutes).

In [9]:
!python -m bank.label --questions bank/pilot/questions_math.jsonl --generations bank/pilot/math/generations.jsonl --out bank/pilot/math/labeled.jsonl
!python -m bank.label_evalplus --questions bank/pilot/questions_code.jsonl --generations bank/pilot/code/generations.jsonl --out bank/pilot/code/labeled.jsonl

4800 answers: {'ok': 4381, 'truncated': 352, 'no_answer': 67}; usable correct=3706, wrong=675
Second checker: math_verify
Load from ground-truth from /root/.cache/evalplus/fe585eb4df8c88d844eeb463ea4d0302.pkl
Load from ground-truth from /root/.cache/evalplus/ee43ecabebf20deef4bb776a405ac5b1.pkl
checked 500/4361 programs
checked 1000/4361 programs
checked 1500/4361 programs
checked 2000/4361 programs
checked 2500/4361 programs
checked 3000/4361 programs
checked 3500/4361 programs
checked 4000/4361 programs
4800 answers: {'ok': 4361, 'no_answer': 413, 'truncated': 26}
usable correct=2606, wrong=1755; error types {'wrong_output': 1755}


## 5. Matched pairs, length filter and the pilot draw

In [10]:
for d in ("math", "code"):
    !python -m bank.make_pairs --split pilot --questions bank/pilot/questions_{d}.jsonl --generations bank/pilot/{d}/generations.jsonl --labeled bank/pilot/{d}/labeled.jsonl --out bank/pilot/{d}
!python -m bank.token_lengths bank/pilot/math/pilot_bank.jsonl bank/pilot/code/pilot_bank.jsonl --out bank/pilot/too_long.json
!python -m bank.sample_pilot --math bank/pilot/math --code bank/pilot/code --too-long bank/pilot/too_long.json --out bank/pilot

No exclusion file at bank/pilot/excluded_questions.txt; no questions excluded.
{
  "cells_with_usable_answers": 1160,
  "cells_excluded": 0,
  "cells_all_correct": 836,
  "cells_all_wrong": 109,
  "pairs": 215,
  "questions": 300,
  "generations": 4800,
  "candidates": 430,
  "excluded_questions": {},
  "exclusions_file": null,
  "exclusions_sha256": null,
  "generator_revisions_from_provenance": null,
  "seed": 544,
  "python": "3.13.16"
}
No exclusion file at bank/pilot/excluded_questions.txt; no questions excluded.
{
  "cells_with_usable_answers": 577,
  "cells_excluded": 0,
  "cells_all_correct": 251,
  "cells_all_wrong": 159,
  "pairs": 167,
  "questions": 150,
  "generations": 4800,
  "candidates": 334,
  "excluded_questions": {},
  "exclusions_file": null,
  "exclusions_sha256": null,
  "generator_revisions_from_provenance": null,
  "seed": 544,
  "python": "3.13.16"
}
config.json: 100% 663/663 [00:00<00:00, 3.28MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 20.3MB/

## 6. Generator accuracy (for the report)

In [11]:
import collections
for d in ("math", "code"):
    lab = {r["candidate_id"]: r for r in map(json.loads, open(f"bank/pilot/{d}/labeled.jsonl", encoding="utf-8"))}
    gens = list(map(json.loads, open(f"bank/pilot/{d}/generations.jsonl", encoding="utf-8")))
    tally = collections.defaultdict(collections.Counter)
    for g in gens:
        l = lab[g["candidate_id"]]
        tally[g["generator"]]["usable" if l["status"] == "ok" else l["status"]] += 1
        tally[g["generator"]]["correct"] += l["label"] == 1
    for gen, c in tally.items():
        print(d, gen, dict(c), "accuracy among usable:", round(c["correct"] / max(1, c["usable"]), 3))

math openai/gpt-4.1-mini {'usable': 1080, 'correct': 1069, 'truncated': 117, 'no_answer': 3} accuracy among usable: 0.99
math ibm-granite/granite-3.3-8b-instruct {'usable': 1077, 'correct': 941, 'truncated': 117, 'no_answer': 6} accuracy among usable: 0.874
math allenai/OLMo-2-1124-7B-Instruct {'usable': 1088, 'correct': 712, 'no_answer': 27, 'truncated': 85} accuracy among usable: 0.654
math microsoft/Phi-4-mini-instruct {'usable': 1136, 'correct': 984, 'no_answer': 31, 'truncated': 33} accuracy among usable: 0.866
code openai/gpt-4.1-mini {'usable': 1181, 'correct': 914, 'truncated': 17, 'no_answer': 2} accuracy among usable: 0.774
code ibm-granite/granite-3.3-8b-instruct {'usable': 1117, 'correct': 654, 'no_answer': 83} accuracy among usable: 0.585
code allenai/OLMo-2-1124-7B-Instruct {'usable': 1036, 'correct': 429, 'no_answer': 164} accuracy among usable: 0.414
code microsoft/Phi-4-mini-instruct {'usable': 1027, 'correct': 609, 'truncated': 9, 'no_answer': 164} accuracy among usab

## Download the results
Zips `bank/pilot/`. Send the zip back, or unzip it into the repo at the same path.

In [12]:
!cd /content/project && zip -qr /content/pilot_bank.zip bank/pilot
from google.colab import files
files.download("/content/pilot_bank.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
!cd /content/project && git pull -q && git log --oneline -1

73497ba (HEAD -> feature/pilot-bank, origin/feature/pilot-bank) Run EvalPlus checks in spawned worker processes, not threads
